# Setup and Explore Databases

## Overview

This notebook is the first step in the lab, designed to finalize the setup of the database environment and to familiarize you with the database schemas and data. The core infrastructure, including an AlloyDB cluster and a Spanner instance, is already provisioned via Terraform. The notebook will guide you through the final setup steps for these databases and allow you to explore their structure and content.

The key milestones you will achieve are:

- Connecting to your Google Cloud project and the provisioned databases.
- Finalizing the setup of the AlloyDB and Spanner databases.
- Importing and validating data in both AlloyDB and Spanner.
- Updating the dates in the datasets to demonstrate real-time capabilities.
- Exploring the schemas and data in both databases through various queries.

### Terraform Resources

The following pre-requisite resources were created for you by Terraform. See the [main.tf](../terraform/main.tf) file for more details on the environment configuration:

- Spanner, AlloyDB, and other related APIs
- Network components for secure communication (Custom VPC, Cloud Router, Cloud NAT Gateway, Firewall Rules, Private IP Range for Service Networking)
- AlloyDB Cluster and Instance
- Spanner Instance, Database, Schema, and Data
- Vertex AI Workbench Instance

### Google Cloud Services Used in this Notebook

This notebook utilizes the following Google Cloud services:
- Vertex AI Workbench: A Jupyter notebook-based development environment for the entire data science workflow.
- AlloyDB: A fully-managed, 100% PostgreSQL-compatible database service to power your most demanding enterprise workloads with superior performance, availability, and scale, and supercharge them with AI.
- Spanner: A fully-managed, geographically distributed relational database with unlimited scale, strong consistency, and up to 99.999% availability, bringing together relational, graph, key value, vector, and search data into a single database.
- Cloud Storage: A managed service for storing any amount of data unstructured data retrieving it as often as you like.
- IAM (Identity and Access Management): Fine-grained access control and visibility for centrally managing cloud resources.

### Logical Flow
This notebook provides step-by-step instructions to guide you through the database setup and exploration process. The logical flow is as follows:
- Basic Setup: This section includes defining notebook variables, connecting to your Google Cloud project, configuring logging, and installing dependencies.
- Helper Functions: Helper functions are defined for interacting with the REST API, AlloyDB, and Spanner to simplify the code in the subsequent sections.
- Finalize AlloyDB Setup: This section connects to the AlloyDB cluster, creates the finance database, and imports data from a SQL file in a GCS bucket. It also includes a step to validate the row counts of the imported tables.
- Update Dates: The dates in both the AlloyDB and Spanner tables are updated to reflect more recent data, which is crucial for demonstrating the real-time capabilities of the solution.
- Explore AlloyDB Data: This section provides a review of the AlloyDB schema and allows you to run queries to explore the data in the various tables.
- Explore Spanner Data: Similar to the AlloyDB section, this part of the notebook reviews the Spanner schema (including the property graph) and allows you to run both relational and graph queries to explore the data.

Let's get started!

## Basic Setup

### Review Organization Policies

If you have [Organization Policies](https://console.cloud.google.com/iam-admin/orgpolicies/list) that explicitly deny certain configurations, you will need to work with your platform team to add an exception. Specifically, you may need to override the following policies with an "Allow All" rule (see [the docs](https://cloud.google.com/resource-manager/docs/organization-policy/creating-managing-policies) for more details): `constraints/run.allowedIngress`, `constraints/iam.allowedPolicyMemberDomains`, `constraints/compute.vmExternalIpAccess`.

### Define Notebook Variables

Update the `project_id` and `region` variables below to match your environment. You can find these values in your Terraform output. You can use defaults for the rest of the project variables .

You will be prompted for two passwords:
1. The AlloyDB admin password for the `postgres` user. This is the password you defined when provisioning the Terraform environment.
2. A new password for a least-privilege `toolbox_user` account that we will create later in the notebook. Take note of this password for later labs.

In [16]:
# Local SQLite configuration (NO CLOUD REQUIRED)

project_id = "local-project"
region = "local"

# No VPC needed in local mode
vpc = None

# Local “bucket placeholder” (not used)
gcs_bucket_name = "local-bucket"

# Database name (used only for labeling)
database_name = "finance"

# No passwords needed for SQLite
print("Running in LOCAL SQLITE MODE (no cloud resources required)")

Running in LOCAL SQLITE MODE (no cloud resources required)


In [17]:
# Local mode: no gRPC or cloud runtime needed
print("Local SQLite mode: no environment variables required")

Local SQLite mode: no environment variables required


### Connect to your Google Cloud Project

In [18]:
# Local mode: no Google Cloud project required
print("Skipping gcloud configuration (running in local SQLite mode)")

Skipping gcloud configuration (running in local SQLite mode)


### Configure Logging

In [19]:
import logging
import sys

# Configure the root logger to output messages with INFO level or above
logging.basicConfig(level=logging.INFO, stream=sys.stdout, format='%(asctime)s[%(levelname)5s][%(name)14s] - %(message)s',  datefmt='%H:%M:%S', force=True)

### Install Dependencies

In [20]:
# Local mode - no cloud dependencies required
print("No external cloud packages needed for SQLite version")

No external cloud packages needed for SQLite version


### Define Helper Functions

#### REST API Helper Function

In [21]:
import json
import requests

# -----------------------------
# SAFE LOCAL / COLAB VERSION
# -----------------------------

# Fake project_id fallback (no GCP required)
project_id = "local-project"

# We do NOT use Google Auth in free mode
authed_session = requests.Session()

# Optional header (kept only for compatibility, harmless locally)
authed_session.headers.update({
    "Content-Type": "application/json",
    "x-goog-user-project": project_id
})


def rest_api_helper(
    url: str,
    http_verb: str,
    request_body: dict = None,
    params: dict = None,
    session: requests.Session = authed_session,
) -> dict:
    """Local-safe REST helper (NO Google auth required)."""

    try:
        if http_verb == "GET":
            response = session.get(url, params=params)

        elif http_verb == "POST":
            response = session.post(url, json=request_body, params=params)

        elif http_verb == "PUT":
            response = session.put(url, json=request_body, params=params)

        elif http_verb == "PATCH":
            response = session.patch(url, json=request_body, params=params)

        elif http_verb == "DELETE":
            response = session.delete(url, params=params)

        else:
            raise ValueError(f"Unknown HTTP verb: {http_verb}")

        response.raise_for_status()

        return response.json() if response.content else {}

    except requests.exceptions.RequestException as e:
        print("Request failed:", e)
        raise RuntimeError(str(e))

    except json.JSONDecodeError:
        print("Failed to decode JSON response")
        return {}

#### AlloyDB Helper Function

In [22]:
import sqlite3
import pandas as pd
import logging

# Create a single local SQLite database file (in-memory or disk)
SQLITE_DB = "local_finance.db"


def run_sqlite_query(sql: str, params=None, output_as_df: bool = True):
    """
    Free replacement for AlloyDB query runner using SQLite.
    Fully synchronous (NO async needed).
    """

    conn = sqlite3.connect(SQLITE_DB)
    cursor = conn.cursor()

    sql_lower = sql.strip().lower()
    is_select = sql_lower.startswith(("select", "with"))

    try:
        # -------------------------
        # PARAMETER HANDLING
        # -------------------------
        if params:
            if isinstance(params, list):
                cursor.executemany(sql, params)   # bulk insert/update
            else:
                cursor.execute(sql, params)
        else:
            cursor.execute(sql)

        # -------------------------
        # SELECT QUERY
        # -------------------------
        if is_select:
            rows = cursor.fetchall()
            cols = [desc[0] for desc in cursor.description]

            if output_as_df:
                return pd.DataFrame(rows, columns=cols)
            else:
                return rows

        # -------------------------
        # NON-SELECT (CREATE/INSERT/UPDATE)
        # -------------------------
        conn.commit()
        return {"status": "success", "rows_affected": cursor.rowcount}

    except Exception as e:
        logging.error(f"SQL error: {e}")
        return {"status": "error", "message": str(e)}

    finally:
        conn.close()

#### Spanner Helper Functions

In [23]:
import sqlite3
import pandas as pd
import time

# -----------------------------------
# LOCAL SQLITE DB FILE
# -----------------------------------
SQLITE_DB = "spanner_sim.db"

# Create session equivalent (dummy)
session = "sqlite-session"


# -----------------------------------
# SESSION FUNCTIONS (NO-OP EQUIVALENT)
# -----------------------------------
def get_spanner_sessions():
    return {"sessions": ["sqlite-session"]}


def create_spanner_session():
    print("Using local SQLite session (no cloud needed).")
    return "sqlite-session"


def close_spanner_session(session):
    print("Closing local SQLite session (no action needed).")
    return {"status": "closed"}


# -----------------------------------
# CORE QUERY RUNNER (REPLACES SPANNER API)
# -----------------------------------
def run_spanner_query(sql, query_options=None, create_new_session=False):

    global session

    if not session or create_new_session:
        session = create_spanner_session()

    conn = sqlite3.connect(SQLITE_DB)
    cursor = conn.cursor()

    sql_lower = sql.strip().lower()
    is_select = sql_lower.startswith(("select", "with", "graph"))

    try:
        cursor.execute(sql)

        # -------------------------
        # SELECT QUERY → DataFrame
        # -------------------------
        if is_select:
            rows = cursor.fetchall()
            columns = [desc[0] for desc in cursor.description]

            if rows:
                return pd.DataFrame(rows, columns=columns)
            else:
                return pd.DataFrame()

        # -------------------------
        # WRITE QUERY
        # -------------------------
        conn.commit()
        return {"status": "success", "rows_affected": cursor.rowcount}

    except Exception as e:
        print("SQL Error:", e)
        return {"status": "error", "message": str(e)}

    finally:
        conn.close()


# -----------------------------------
# DDL HANDLER (CREATE TABLES ETC.)
# -----------------------------------
def run_spanner_ddl(ddl_array):

    conn = sqlite3.connect(SQLITE_DB)
    cursor = conn.cursor()

    try:
        for ddl in ddl_array:
            cursor.execute(ddl)

        conn.commit()
        print("DDL executed successfully.")
        return {"status": "success"}

    except Exception as e:
        print("DDL Error:", e)
        return {"status": "error", "message": str(e)}

    finally:
        conn.close()

## Finalize AlloyDB Setup

### Connect to the AlloyDB Cluster

This function will create a connection pool to your AlloyDB instance using the AlloyDB Python connector. The AlloyDB Python connector will automatically create secure connections to your AlloyDB instance using mTLS.

In [24]:
import sqlite3

# -----------------------------
# SQLITE DATABASE FILE
# -----------------------------
SQLITE_DB = "local_finance.db"

# -----------------------------
# "POOL" REPLACEMENT (FAKE)
# -----------------------------
# We keep these names ONLY so rest of notebook doesn't break
class FakePool:
    def __init__(self, db_name):
        self.db_name = db_name

    def connect(self):
        return sqlite3.connect(SQLITE_DB)


# -----------------------------
# REPLACE ALLLOYDB POOLS
# -----------------------------
postgres_db_pool = FakePool("postgres")
finance_db_pool = FakePool("finance")


# -----------------------------
# OPTIONAL: INIT TABLES FILE
# -----------------------------
def init_sqlite():
    conn = sqlite3.connect(SQLITE_DB)
    cursor = conn.cursor()

    # You can pre-create common structures if needed
    # (safe even if repeated)
    cursor.execute("PRAGMA foreign_keys = ON;")

    conn.commit()
    conn.close()


init_sqlite()

print("SQLite environment ready (no AlloyDB, no billing needed).")

SQLite environment ready (no AlloyDB, no billing needed).


### Create AlloyDB Database

In [25]:
sql = "SELECT sqlite_version();"
result = run_sqlite_query(sql)
print(result)

  sqlite_version()
0           3.37.2


In [26]:
!wget -O finance.sql https://storage.googleapis.com/pr-public-demo-data/adk-toolbox-demo/data/finance.sql

--2026-04-22 14:51:21--  https://storage.googleapis.com/pr-public-demo-data/adk-toolbox-demo/data/finance.sql
Resolving storage.googleapis.com (storage.googleapis.com)... 142.250.31.207, 172.253.63.207, 142.251.111.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.250.31.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1351955275 (1.3G) [application/x-sql]
Saving to: ‘finance.sql’

finance.sql         100%[===================>]   1.26G  85.4MB/s    in 16s     

2026-04-22 14:51:37 (82.1 MB/s) - ‘finance.sql’ saved [1351955275/1351955275]



### Import AlloyDB Data

#### Run the Import

This step generally takes about 5 minutes to complete.

In [35]:
import sqlite3

SQLITE_DB = "local_finance.db"

conn = sqlite3.connect(SQLITE_DB)
cursor = conn.cursor()

# Create all tables that your SQL expects
tables_to_create = {
    "users": "CREATE TABLE IF NOT EXISTS users (id INTEGER PRIMARY KEY, name TEXT)",
    "mcc_codes": "CREATE TABLE IF NOT EXISTS mcc_codes (id INTEGER PRIMARY KEY, code TEXT)",
    "transactions": "CREATE TABLE IF NOT EXISTS transactions (id INTEGER PRIMARY KEY, user_id INTEGER, amount REAL)",
    "fraud_labels": "CREATE TABLE IF NOT EXISTS fraud_labels (id INTEGER PRIMARY KEY, transaction_id INTEGER, label TEXT)",
    "cards": "CREATE TABLE IF NOT EXISTS cards (id INTEGER PRIMARY KEY, card_number TEXT, user_id INTEGER)"
}

for table_name, ddl in tables_to_create.items():
    cursor.execute(ddl)

conn.commit()
conn.close()

print("All missing tables created (empty) in SQLite.")

All missing tables created (empty) in SQLite.


#### Validate Row Counts

In [36]:
sql = """
SELECT 'users' AS table_name, (SELECT COUNT(*) FROM users) AS imported_count, 2000 AS target_row_count
UNION ALL
SELECT 'mcc_codes', (SELECT COUNT(*) FROM mcc_codes), 109
UNION ALL
SELECT 'transactions', (SELECT COUNT(*) FROM transactions), 13305915
UNION ALL
SELECT 'fraud_labels', (SELECT COUNT(*) FROM fraud_labels), 8914963
UNION ALL
SELECT 'cards', (SELECT COUNT(*) FROM cards), 6146;
"""

# Run using the SQLite query runner
result = run_sqlite_query(sql)
print(result)

     table_name  imported_count  target_row_count
0         users               0              2000
1     mcc_codes               0               109
2  transactions               0          13305915
3  fraud_labels               0           8914963
4         cards               0              6146


## Configure Least Privilege

### Create a Least Privilege AlloyDB User for MCP Toolbox

MCP Toolbox will use the `toolbox_user` principal to securely access the AlloyDB database. This will be a read-only user to ensure so inadvertent data or object changes are made with our Agent.

In [37]:
# ---------------------------------------------------------
# SQLite does not support users, roles, or permissions.
# This cell replaces AlloyDB user creation commands with a no-op.
# ---------------------------------------------------------

# Optional: print a message so you know this step is skipped
print("Skipping AlloyDB user creation and permissions – not needed in SQLite.")

Skipping AlloyDB user creation and permissions – not needed in SQLite.


### Spanner Least Privilege for MCP Toolbox

MCP Toolbox will use the credentials of the Cloud Run service account that hosts it to access Spanner. We created a dedicated service account called `toolbox-service-account` via Terraform with the `roles/databaseUser` role for this purpose.

## Update Dates

Our data set is several years old. We'll update the dates to show off the real-time capabilities of leveraging MCP Toolbox with ADK.

### Update AlloyDB Dates

#### Update Transaction Dates

> NOTE: This update may take 15-20 minutes as there are over 13 Million rows to update in the transactions table.

In [39]:
import sqlite3
import pandas as pd

SQLITE_DB = "local_finance.db"
sql = "SELECT MAX(date) AS max_date, MIN(date) AS min_date FROM transactions;"

conn = sqlite3.connect(SQLITE_DB)
cursor = conn.cursor()

try:
    # Check if table exists
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='transactions';")
    if cursor.fetchone() is None:
        print("Table 'transactions' does not exist. Skipping MAX/MIN date query.")
        df_result = pd.DataFrame([{"max_date": None, "min_date": None}])
    else:
        # Check if column exists
        cursor.execute("PRAGMA table_info(transactions);")
        columns = [col[1] for col in cursor.fetchall()]
        if "date" not in columns:
            print("Column 'date' does not exist in 'transactions'.")
            df_result = pd.DataFrame([{"max_date": None, "min_date": None}])
        else:
            cursor.execute(sql)
            row = cursor.fetchone()
            df_result = pd.DataFrame([row], columns=[desc[0] for desc in cursor.description])
finally:
    conn.close()

print(df_result)

Column 'date' does not exist in 'transactions'.
  max_date min_date
0     None     None


In [40]:
import sqlite3
from datetime import datetime, timedelta

SQLITE_DB = "local_finance.db"

conn = sqlite3.connect(SQLITE_DB)
cursor = conn.cursor()

# Check if transactions table and date column exist
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='transactions';")
if cursor.fetchone() is not None:
    cursor.execute("PRAGMA table_info(transactions);")
    columns = [col[1] for col in cursor.fetchall()]
    if "date" in columns:
        # Get the max date in the table
        cursor.execute("SELECT MAX(date) FROM transactions;")
        max_date_str = cursor.fetchone()[0]
        if max_date_str is not None:
            # Convert to datetime
            max_date = datetime.fromisoformat(max_date_str)
            now = datetime.now()

            # Update all rows proportionally
            cursor.execute("SELECT rowid, date FROM transactions ORDER BY date;")
            rows = cursor.fetchall()

            for rowid, date_str in rows:
                old_date = datetime.fromisoformat(date_str)
                delta = max_date - old_date
                new_date = now - delta
                cursor.execute("UPDATE transactions SET date = ? WHERE rowid = ?", (new_date.isoformat(), rowid))

            conn.commit()
            print(f"Shifted {len(rows)} transaction dates forward to simulate real-time.")
        else:
            print("No dates found in transactions table.")
    else:
        print("Column 'date' does not exist in transactions table.")
else:
    print("Table 'transactions' does not exist.")

conn.close()

Column 'date' does not exist in transactions table.


In [41]:
import sqlite3
import pandas as pd

SQLITE_DB = "local_finance.db"
sql = "SELECT MAX(date) AS max_date, MIN(date) AS min_date FROM transactions;"

conn = sqlite3.connect(SQLITE_DB)
cursor = conn.cursor()

try:
    # Check if transactions table exists
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='transactions';")
    if cursor.fetchone() is None:
        print("Table 'transactions' does not exist. Returning empty results.")
        df_result = pd.DataFrame([{"max_date": None, "min_date": None}])
    else:
        # Check if date column exists
        cursor.execute("PRAGMA table_info(transactions);")
        columns = [col[1] for col in cursor.fetchall()]
        if "date" not in columns:
            print("Column 'date' does not exist in 'transactions'. Returning empty results.")
            df_result = pd.DataFrame([{"max_date": None, "min_date": None}])
        else:
            # Execute the query
            cursor.execute(sql)
            row = cursor.fetchone()
            df_result = pd.DataFrame([row], columns=[desc[0] for desc in cursor.description])
finally:
    conn.close()

print(df_result)

Column 'date' does not exist in 'transactions'. Returning empty results.
  max_date min_date
0     None     None


#### Update Card Dates

In [42]:
import sqlite3
import pandas as pd

SQLITE_DB = "local_finance.db"

sql = """
SELECT MAX(acct_open_date) AS max_acct_open_date,
       MIN(acct_open_date) AS min_acct_open_date,
       MAX(year_pin_last_changed) AS max_year_pin_last_changed,
       MIN(year_pin_last_changed) AS min_year_pin_last_changed
FROM cards;
"""

conn = sqlite3.connect(SQLITE_DB)
cursor = conn.cursor()

try:
    # Check if 'cards' table exists
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='cards';")
    if cursor.fetchone() is None:
        print("Table 'cards' does not exist. Returning empty results.")
        df_result = pd.DataFrame([{
            "max_acct_open_date": None,
            "min_acct_open_date": None,
            "max_year_pin_last_changed": None,
            "min_year_pin_last_changed": None
        }])
    else:
        # Check if required columns exist
        cursor.execute("PRAGMA table_info(cards);")
        columns = [col[1] for col in cursor.fetchall()]
        required_cols = ["acct_open_date", "year_pin_last_changed"]
        missing_cols = [c for c in required_cols if c not in columns]
        if missing_cols:
            print(f"Columns {missing_cols} do not exist in 'cards'. Returning empty results.")
            df_result = pd.DataFrame([{
                "max_acct_open_date": None,
                "min_acct_open_date": None,
                "max_year_pin_last_changed": None,
                "min_year_pin_last_changed": None
            }])
        else:
            # Execute the query
            cursor.execute(sql)
            row = cursor.fetchone()
            df_result = pd.DataFrame([row], columns=[desc[0] for desc in cursor.description])
finally:
    conn.close()

print(df_result)

Columns ['acct_open_date', 'year_pin_last_changed'] do not exist in 'cards'. Returning empty results.
  max_acct_open_date min_acct_open_date max_year_pin_last_changed  \
0               None               None                      None   

  min_year_pin_last_changed  
0                      None  


In [ ]:
# This query shifts all transaction dates forward.
# The most recent record will be updated to the current time (NOW()),
# and all other record will be adjusted proportionally,
# preserving the original time intervals between them.
sql = """
UPDATE cards
SET
    acct_open_date = (NOW() - ((SELECT MAX(acct_open_date) FROM cards) - acct_open_date) * INTERVAL '1 day')::date,
    year_pin_last_changed = year_pin_last_changed + (EXTRACT(YEAR FROM NOW())::INTEGER - (SELECT MAX(year_pin_last_changed) FROM cards));
"""
await run_alloydb_query(finance_db_pool, sql)

In [ ]:
# See new MAX and MIN dates
sql = """
SELECT MAX(acct_open_date) AS max_acct_open_date,
    MIN(acct_open_date) AS min_acct_open_date,
    MAX(year_pin_last_changed) AS max_year_pin_last_changed,
    MIN(year_pin_last_changed) AS min_year_pin_last_changed
FROM cards;"""
await run_alloydb_query(finance_db_pool, sql)

### Update Spanner Dates

#### Verify Spanner Row Counts

> IMPORTANT: If data is missing from Spanner, check the status of the `import-spanner` job in [Dataflow](https://console.cloud.google.com/dataflow/jobs). Resolve any errors and re-run the job before proceeding.

In [ ]:
sql = """
SELECT 'Account' AS table_name, (SELECT COUNT(*) FROM Account) AS row_count, 700 AS target_row_count
UNION ALL
SELECT 'Loan', (SELECT COUNT(*) FROM Loan), 500
UNION ALL
SELECT 'Person', (SELECT COUNT(*) FROM Person), 500
UNION ALL
SELECT 'AccountTransferAccount', (SELECT COUNT(*) FROM AccountTransferAccount), 500
UNION ALL
SELECT 'AccountRepayLoan', (SELECT COUNT(*) FROM AccountRepayLoan), 500
UNION ALL
SELECT 'PersonOwnAccount', (SELECT COUNT(*) FROM PersonOwnAccount), 700
UNION ALL
SELECT 'AccountAudits', (SELECT COUNT(*) FROM AccountAudits), 500;
"""

result = run_spanner_query(sql)
result

#### View Current Min and Max Dates

In [ ]:
sql = """
SELECT 'Account' AS table_name, MAX(create_time) AS max_time, MIN(create_time) AS min_time FROM Account
UNION ALL
SELECT 'AccountAudits', MAX(audit_timestamp), MIN(audit_timestamp) FROM AccountAudits
UNION ALL
SELECT 'AccountRepayLoan', MAX(create_time), MIN(create_time) FROM AccountRepayLoan
UNION ALL
SELECT 'AccountTransferAccount', MAX(create_time), MIN(create_time) FROM AccountTransferAccount
UNION ALL
SELECT 'Loan', MAX(create_time), MIN(create_time) FROM Loan
UNION ALL
SELECT 'PersonOwnAccount', MAX(create_time), MIN(create_time) FROM PersonOwnAccount;
"""

run_spanner_query(sql)

#### Update Date Columns

In [ ]:
sql_array = []

sql_array.append("""
UPDATE Account
SET create_time = TIMESTAMP_SUB(
  CURRENT_TIMESTAMP(),
  INTERVAL TIMESTAMP_DIFF(
    (SELECT MAX(t.create_time) FROM Account AS t),
    create_time,
    MICROSECOND
  ) MICROSECOND
)
WHERE create_time IS NOT NULL;
""")

sql_array.append("""
UPDATE Loan
SET create_time = TIMESTAMP_SUB(
  CURRENT_TIMESTAMP(),
  INTERVAL TIMESTAMP_DIFF(
    (SELECT MAX(t.create_time) FROM Loan AS t),
    create_time,
    MICROSECOND
  ) MICROSECOND
)
WHERE create_time IS NOT NULL;
""")

sql_array.append("""
UPDATE PersonOwnAccount
SET create_time = TIMESTAMP_SUB(
  CURRENT_TIMESTAMP(),
  INTERVAL TIMESTAMP_DIFF(
    (SELECT MAX(t.create_time) FROM PersonOwnAccount AS t),
    create_time,
    MICROSECOND
  ) MICROSECOND
)
WHERE create_time IS NOT NULL;
""")

for sql in sql_array:
    result = run_spanner_query(sql)
    print(f"{result}\n")

#### Handle Primary Key Column Updates

This code handles a special case where timestampe columns are part of the Primary Key on three of our tables. Spanner does not allow updates on key columns. To work around this, we will:
1. Create temporary tables with the same schema.
2. Copy the data from the original tables to the temporary tables, updating the timestamps in the process.
3. Delete the data from the original tables.
4. Reload the data from the temporary tables back into the original tables.
5. Drop the temporary tables.

#### Create Temp Tables

In [ ]:
ddl_array = []

ddl_array.append("""
CREATE TABLE AccountAuditsTemp (
  id INT64 NOT NULL,
  audit_timestamp TIMESTAMP,
  audit_details STRING(MAX),
) PRIMARY KEY(id, audit_timestamp)
""")

ddl_array.append("""
CREATE TABLE AccountRepayLoanTemp (
  id INT64 NOT NULL,
  loan_id INT64 NOT NULL,
  amount FLOAT64,
  create_time TIMESTAMP NOT NULL,
) PRIMARY KEY(id, loan_id, create_time)
""")

ddl_array.append("""
CREATE TABLE AccountTransferAccountTemp (
  id INT64 NOT NULL,
  to_id INT64 NOT NULL,
  amount FLOAT64,
  create_time TIMESTAMP NOT NULL,
) PRIMARY KEY(id, to_id, create_time)
""")

run_spanner_ddl(ddl_array)

In [ ]:
# Insert data into temp tables with new timestamps
sql_array = []

sql_array.append(
"""
INSERT INTO AccountAuditsTemp (id, audit_timestamp, audit_details)
SELECT
    id,
    TIMESTAMP_SUB(
        CURRENT_TIMESTAMP(),
        INTERVAL TIMESTAMP_DIFF(
            (SELECT MAX(t.audit_timestamp) FROM AccountAudits AS t),
            audit_timestamp,
            MICROSECOND
        ) MICROSECOND
    ) AS audit_timestamp,
    audit_details
FROM
    AccountAudits
"""
)

sql_array.append("""
INSERT INTO AccountRepayLoanTemp (id, loan_id, amount, create_time)
SELECT
    id,
    loan_id,
    amount,
    TIMESTAMP_SUB(
        CURRENT_TIMESTAMP(),
        INTERVAL TIMESTAMP_DIFF(
            (SELECT MAX(t.create_time) FROM AccountRepayLoan AS t),
            create_time,
            MICROSECOND
        ) MICROSECOND
    ) AS create_time
FROM
    AccountRepayLoan
""")

sql_array.append("""
INSERT INTO AccountTransferAccountTemp (id, to_id, amount, create_time)
SELECT
    id,
    to_id,
    amount,
    TIMESTAMP_SUB(
        CURRENT_TIMESTAMP(),
        INTERVAL TIMESTAMP_DIFF(
            (SELECT MAX(t.create_time) FROM AccountTransferAccount AS t),
            create_time,
            MICROSECOND
        ) MICROSECOND
    ) AS create_time
FROM
    AccountTransferAccount
""")

for sql in sql_array:
    run_spanner_query(sql)

In [ ]:
# View timestamp data in temp tables

sql = """
SELECT 'AccountAuditsTemp' AS table_name, MAX(audit_timestamp) AS max_time, MIN(audit_timestamp) AS min_time FROM AccountAuditsTemp
UNION ALL
SELECT 'AccountRepayLoanTemp', MAX(create_time), MIN(create_time) FROM AccountRepayLoanTemp
UNION ALL
SELECT 'AccountTransferAccountTemp', MAX(create_time), MIN(create_time) FROM AccountTransferAccountTemp
"""
run_spanner_query(sql)

#### Reload Data into Source Tables

In [ ]:
# Delete data in source tables
sql_array = []

sql_array.append("DELETE FROM AccountAudits WHERE 1 = 1")

sql_array.append("DELETE FROM AccountRepayLoan WHERE 1 = 1")

sql_array.append("DELETE FROM AccountTransferAccount WHERE 1 = 1")

for sql in sql_array:
    run_spanner_query(sql)

In [ ]:
# Load source tables from temp tables

sql_array = []

sql_array.append(
"""
INSERT INTO AccountAudits (id, audit_timestamp, audit_details)
SELECT
    id,
    audit_timestamp,
    audit_details
FROM
    AccountAuditsTemp
"""
)

sql_array.append("""
INSERT INTO AccountRepayLoan (id, loan_id, amount, create_time)
SELECT
    id,
    loan_id,
    amount,
    create_time
FROM
    AccountRepayLoanTemp
""")

sql_array.append("""
INSERT INTO AccountTransferAccount (id, to_id, amount, create_time)
SELECT
    id,
    to_id,
    amount,
    create_time
FROM
    AccountTransferAccountTemp
""")

for sql in sql_array:
    run_spanner_query(sql)

#### View New Min and Max Dates

In [ ]:
sql = """
SELECT 'Account' AS table_name, MAX(create_time) AS max_time, MIN(create_time) AS min_time FROM Account
UNION ALL
SELECT 'AccountAudits', MAX(audit_timestamp), MIN(audit_timestamp) FROM AccountAudits
UNION ALL
SELECT 'AccountRepayLoan', MAX(create_time), MIN(create_time) FROM AccountRepayLoan
UNION ALL
SELECT 'AccountTransferAccount', MAX(create_time), MIN(create_time) FROM AccountTransferAccount
UNION ALL
SELECT 'Loan', MAX(create_time), MIN(create_time) FROM Loan
UNION ALL
SELECT 'PersonOwnAccount', MAX(create_time), MIN(create_time) FROM PersonOwnAccount;
"""

run_spanner_query(sql)

#### Clean Up Temp Tables

In [ ]:
ddl_array = []

ddl_array.append("""
DROP TABLE AccountAuditsTemp
""")

ddl_array.append("""
DROP TABLE AccountRepayLoanTemp
""")

ddl_array.append("""
DROP TABLE AccountTransferAccountTemp
""")

run_spanner_ddl(ddl_array)

#### Verify Spanner Row Counts

In [ ]:
sql = """
SELECT 'Account' AS table_name, (SELECT COUNT(*) FROM Account) AS row_count, 700 AS target_row_count
UNION ALL
SELECT 'Loan', (SELECT COUNT(*) FROM Loan), 500
UNION ALL
SELECT 'Person', (SELECT COUNT(*) FROM Person), 500
UNION ALL
SELECT 'AccountTransferAccount', (SELECT COUNT(*) FROM AccountTransferAccount), 500
UNION ALL
SELECT 'AccountRepayLoan', (SELECT COUNT(*) FROM AccountRepayLoan), 500
UNION ALL
SELECT 'PersonOwnAccount', (SELECT COUNT(*) FROM PersonOwnAccount), 700
UNION ALL
SELECT 'AccountAudits', (SELECT COUNT(*) FROM AccountAudits), 500;
"""

result = run_spanner_query(sql)
result

## Explore AlloyDB Data

### Review AlloyDB Schema

The AlloyDB database contains data related to transactions, credit cards, users, mcc codes, and historical fraud cases. See the ERD diagram below for details on the tables, columns, and relationships.

![AlloyDB Schema](https://github.com/GoogleCloudPlatform/training-data-analyst/blob/master/courses/ai-agents-with-databases/notebooks/img/alloydb_finance_db_erd.png?raw=1)

### Run AlloyDB Queries

In [ ]:
sql = "SELECT * FROM transactions LIMIT 10;"
await run_alloydb_query(finance_db_pool, sql)

In [ ]:
sql = "SELECT * FROM cards LIMIT 10;"
await run_alloydb_query(finance_db_pool, sql)

In [ ]:
sql = "SELECT * FROM users LIMIT 10;"
await run_alloydb_query(finance_db_pool, sql)

In [ ]:
sql = "SELECT * FROM mcc_codes LIMIT 10;"
await run_alloydb_query(finance_db_pool, sql)

In [ ]:
sql = "SELECT * FROM fraud_labels LIMIT 10;"
await run_alloydb_query(finance_db_pool, sql)

## Explore Spanner Data

### Review the Spanner Schema

The Spanner dataset is already loaded for you via Terraform automation. It contains data about accounts, account transfers, account audits, loans, and loan repayments.

The schema is defined as follows:

```
CREATE TABLE Account (
  id INT64 NOT NULL,
  create_time TIMESTAMP,
  is_blocked BOOL,
  type STRING(MAX),
) PRIMARY KEY(id);

CREATE TABLE AccountAudits (
  id INT64 NOT NULL,
  audit_timestamp TIMESTAMP,
  audit_details STRING(MAX),
) PRIMARY KEY(id, audit_timestamp),
  INTERLEAVE IN PARENT Account ON DELETE CASCADE;

CREATE TABLE AccountRepayLoan (
  id INT64 NOT NULL,
  loan_id INT64 NOT NULL,
  amount FLOAT64,
  create_time TIMESTAMP NOT NULL,
) PRIMARY KEY(id, loan_id, create_time),
  INTERLEAVE IN PARENT Account ON DELETE CASCADE;

CREATE TABLE AccountTransferAccount (
  id INT64 NOT NULL,
  to_id INT64 NOT NULL,
  amount FLOAT64,
  create_time TIMESTAMP NOT NULL,
) PRIMARY KEY(id, to_id, create_time),
  INTERLEAVE IN PARENT Account ON DELETE CASCADE;

CREATE TABLE Loan (
  id INT64 NOT NULL,
  loan_amount FLOAT64,
  balance FLOAT64,
  create_time TIMESTAMP,
  interest_rate FLOAT64,
) PRIMARY KEY(id);

CREATE TABLE Person (
  id INT64 NOT NULL,
  name STRING(MAX),
) PRIMARY KEY(id);

CREATE TABLE PersonOwnAccount (
  id INT64 NOT NULL,
  account_id INT64 NOT NULL,
  create_time TIMESTAMP,
) PRIMARY KEY(id, account_id),
  INTERLEAVE IN PARENT Person ON DELETE CASCADE;

CREATE PROPERTY GRAPH FinGraph
  NODE TABLES(
    Account
      KEY(id)
      LABEL Account PROPERTIES(
        create_time,
        id,
        is_blocked,
        type),

    Loan
      KEY(id)
      LABEL Loan PROPERTIES(
        balance,
        create_time,
        id,
        interest_rate,
        loan_amount),

    Person
      KEY(id)
      LABEL Person PROPERTIES(
        id,
        name)
  )
  EDGE TABLES(
    AccountRepayLoan
      KEY(id, loan_id, create_time)
      SOURCE KEY(id) REFERENCES Account(id)
      DESTINATION KEY(loan_id) REFERENCES Loan(id)
      LABEL Repays PROPERTIES(
        amount,
        create_time,
        id,
        loan_id),

    AccountTransferAccount
      KEY(id, to_id, create_time)
      SOURCE KEY(id) REFERENCES Account(id)
      DESTINATION KEY(to_id) REFERENCES Account(id)
      LABEL Transfers PROPERTIES(
        amount,
        create_time,
        id,
        to_id),

    PersonOwnAccount
      KEY(id, account_id)
      SOURCE KEY(id) REFERENCES Person(id)
      DESTINATION KEY(account_id) REFERENCES Account(id)
      LABEL Owns PROPERTIES(
        account_id,
        create_time,
        id)
  );
```

Notice that the Spanner schema takes advantage of both the core relational database model, as well as Spanner's built-in property graph model. The high-level graph nodes and edges are visualized below.

![Spanner Graph Schema](https://github.com/GoogleCloudPlatform/training-data-analyst/blob/master/courses/ai-agents-with-databases/notebooks/img/spanner_graph_schema.png?raw=1)

### Run Relational Spanner Queries

In [ ]:
sql = "SELECT * FROM Account LIMIT 10;"
run_spanner_query(sql)

In [ ]:
sql = "SELECT * FROM AccountAudits LIMIT 10;"
run_spanner_query(sql)

In [ ]:
sql = "SELECT * FROM AccountRepayLoan LIMIT 10;"
run_spanner_query(sql)

In [ ]:
sql = "SELECT * FROM AccountTransferAccount LIMIT 10;"
run_spanner_query(sql)

In [ ]:
sql = "SELECT * FROM Loan LIMIT 10;"
run_spanner_query(sql)

In [ ]:
sql = "SELECT * FROM Person LIMIT 10;"
run_spanner_query(sql)

In [ ]:
sql = "SELECT * FROM Account LIMIT 10;"
run_spanner_query(sql)

### Run Spanner Graph Queries

Read more about running graph queries on Spanner in the [Spanner docs](https://cloud.google.com/spanner/docs/graph/queries-overview).

In [ ]:
sql = """
GRAPH FinGraph
MATCH
  (person:Person {name: "Jacoby"})-[own:Owns]->
  (account:Account)-[repay:Repays]->(loan:Loan)
RETURN
  account.id AS account_id,
  repay.create_time AS repay_time,
  repay.amount AS loan_repay_amount,
  loan.id AS loan_id,
  loan.loan_amount AS loan_amount
ORDER BY repay.create_time;
"""
run_spanner_query(sql)

In [ ]:
sql = """
GRAPH FinGraph
MATCH ANY SHORTEST
  (src_accnt:Account {id:75} )-[transfers:Transfers]->{3,6}
  (dst_accnt:Account {id:199})
RETURN
  ARRAY_LENGTH(transfers) AS num_hops,
  TO_JSON(transfers) AS transfer_edges;
"""

run_spanner_query(sql)

Congratulations, you have completed Module 1! Proceed to [`2_deploy_mcp_toolbox.ipynb`](./2_deploy_mcp_toolbox.ipynb) to configure and deploy MCP Toolbox.